In [ ]:
import torch

from onomatron.data import Vocab, load_names, make_ngram_pairs, split_names
from onomatron.utils import lr_range_test, plot_history, plot_lr_range, train
from onomatron.wavenet import WaveNet

%matplotlib inline

## Load names, split train/val/test, build vocab

In [ ]:
names = load_names("../data/processed/names.txt")
train_names, val_names, test_names = split_names(names, seed=42)
vocab = Vocab(train_names)  # vocab from train only: no leakage from val/test
print(
    f"{len(names)} names -> {len(train_names)} train / {len(val_names)} val"
    f" / {len(test_names)} test, vocab of {len(vocab)} chars"
)

In [ ]:
block_size = 8


def contexts(name_list: list[str]) -> tuple[torch.Tensor, torch.Tensor]:
    pairs = make_ngram_pairs(
        name_list, vocab, block_size + 1
    )  # (N, block_size+1), dernière col = cible
    return pairs[:, :-1], pairs[:, -1]


X, Y = contexts(train_names)
X_val, Y_val = contexts(val_names)
X_test, Y_test = contexts(test_names)
print(f"{X.shape[0]} train / {X_val.shape[0]} val / {X_test.shape[0]} test contexts")

## Build WaveNet

In [ ]:
emb_dim, h_dim = 2, 100

seed = 42
model = WaveNet(vocab, block_size, emb_dim, h_dim, seed)
print(f"Wavenet model has {model.n_parameters} parameters.")

## Find a maximum learning rate

In [ ]:
# Pick lr (full-batch, can take a while)
lrs = (10 ** torch.linspace(-3, 1, 100)).tolist()
losses = lr_range_test(
    lambda s: WaveNet(vocab, block_size, emb_dim, h_dim, seed=s),
    X,
    Y,
    lrs,
    batch_size=32,
)
best_lr = plot_lr_range(lrs, losses) / 10
print(f"Maximum learning rate found is {best_lr}")

## Train on train, watch val

In [ ]:
n_steps = 100000
batch_size = 32
lr = best_lr

model = WaveNet(vocab, block_size, emb_dim, h_dim, seed)
opt = torch.optim.AdamW(model.parameters(), lr=best_lr)
sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=lr, total_steps=n_steps)

history = train(
    model,
    X,
    Y,
    opt,
    steps=n_steps,
    batch_size=batch_size,
    scheduler=sched,
    seed=42,
    x_val=X_val,
    y_val=Y_val,
    eval_every=100,
    clip_grad=1.0,
)
plot_history(history, smooth=20)
print(
    f"Final train loss {history.train_losses[-1]:.4f}"
    f" | val NLL {history.val[-1][1]:.4f}"
)

## Sample from the model (min 3 chars)

In [ ]:
model.sample(20)

## Final evaluation on test

In [ ]:
print(
    f"MLP: train NLL {model.nll(X, Y):.4f}"
    f" | val NLL {model.nll(X_val, Y_val):.4f}"
    f" | test NLL {model.nll(X_test, Y_test):.4f}"
)